# Pretrained models — one registry for every family

Epione loads published weights for several model families: ChromBPNet's
ENCODE releases, PRINT's footprinting networks, HINT-ATAC's bias tables and
HMMs, Borzoi backbones, the published scooby models and Epione's own
seq2PRINT checkpoints. They are all entries of a single registry,
`epi.models.registry`:

```python
epi.models.list_pretrained(family=None, **filters)   # -> DataFrame
epi.models.load_pretrained(name, **loader_kwargs)    # -> model
epi.models.fetch_pretrained(name)                    # -> verified local file(s)
epi.models.register(entry)                           # add your own
```

Each entry records what the model is (family, task, organism, genome,
biosample), where it comes from — a Hugging Face repository **at a pinned
commit**, an ENCODE accession, a Zenodo record, a Git commit — every file
with an **md5 or sha256**, its license and citation, and the loader that
builds it. Nothing is re-hosted: files are downloaded from the original
publisher on first use, checked against the pinned checksum and cached. A
file that no longer matches is refused rather than silently loaded, so a
script keeps loading the model it was written against.

The per-family functions still work and now go through the registry:
`epi.models.chrombpnet.load_pretrained('encode_k562', 'bias')`,
`epi.models.seq2print.load_pretrained('p2s1_fold0_bulk')`,
`epi.tl.fetch_print_model('tn5_bias')`, `epi.tl.fetch_hint_file(...)`.

## Setup

In [1]:
import os, sys, pathlib


def _find_repo(start):
    p = pathlib.Path(start).resolve()
    for cand in (p, *p.parents):
        if (cand / 'epione' / '__init__.py').exists():
            return cand
    return None


CODE = (pathlib.Path(os.environ['EPIONE_SRC']) if os.environ.get('EPIONE_SRC')
        else _find_repo(pathlib.Path.cwd()))
if CODE is not None:
    sys.path.insert(0, str(CODE))

import warnings
warnings.filterwarnings('ignore', module='tqdm')
import numpy as np
import pandas as pd
import epione as epi

pd.set_option('display.max_colwidth', 60)

## What is registered

Listing needs no torch and touches no network. `cached` says whether the
files are already on this machine.

In [2]:
models = epi.models.list_pretrained()
models[['family', 'task', 'organism', 'genome', 'biosample', 'source', 'size_mb', 'license', 'cached']]

,family,task,organism,genome,biosample,source,size_mb,license,cached
name,,,,,,,,,
seq2print_p2s1_fold0_bulk,seq2print,sequence -> multi-scale footprints,mouse,mm10,"mouse brain section P2S1 (spatial ATAC), all spots pooled",hf:starlitnightly/epione-seq2print/p2s1_fold0_bulk@e8a36d4,47.20,GPL-3.0,True
seq2print_p2s1_fold0_lora_bands,seq2print,sequence -> multi-scale footprints,mouse,mm10,"mouse brain section P2S1 (spatial ATAC), four spatial bands",hf:starlitnightly/epione-seq2print/p2s1_fold0_lora_bands...,152.20,GPL-3.0,False
chrombpnet_encode_gm12878,chrombpnet,sequence -> base-resolution ATAC profile + counts,human,GRCh38,GM12878,encode:ENCFF772UBB,504.10,ENCODE data-use policy (free reuse with citation); chrom...,False
chrombpnet_encode_k562,chrombpnet,sequence -> base-resolution ATAC profile + counts,human,GRCh38,K562,encode:ENCFF008TMU,508.10,ENCODE data-use policy (free reuse with citation); chrom...,True
chrombpnet_encode_hepg2,chrombpnet,sequence -> base-resolution ATAC profile + counts,human,GRCh38,HepG2,encode:ENCFF839MZM,506.20,ENCODE data-use policy (free reuse with citation); chrom...,False
chrombpnet_encode_imr90,chrombpnet,sequence -> base-resolution ATAC profile + counts,human,GRCh38,IMR-90,encode:ENCFF419JCT,595.50,ENCODE data-use policy (free reuse with citation); chrom...,False
chrombpnet_encode_tcell,chrombpnet,sequence -> base-resolution ATAC profile + counts,human,GRCh38,T-cell,encode:ENCFF358NCZ,505.90,ENCODE data-use policy (free reuse with citation); chrom...,False
print_dispersion,print,footprint null (dispersion MLP),any,any,cell-type agnostic,zenodo:15224770,0.74,CC-BY-4.0,True
print_tn5_bias,print,Tn5 sequence bias,any,any,cell-type agnostic,zenodo:15224770,0.10,CC-BY-4.0,True


Filters are column names. Strings match case-insensitively, a list matches
any member, a function is applied to the column:

In [3]:
epi.models.list_pretrained('chrombpnet', biosample=['K562', 'hepg2'])[['biosample', 'source', 'size_mb']]

,biosample,source,size_mb
name,,,
chrombpnet_encode_k562,K562,encode:ENCFF008TMU,508.1
chrombpnet_encode_hepg2,HepG2,encode:ENCFF839MZM,506.2


In [4]:
epi.models.list_pretrained(organism='human', size_mb=lambda mb: mb < 10)[['family', 'task', 'source']]

,family,task,source
name,,,
hint_atac_bias_tables,hint,Tn5 sequence bias (8-mer tables),github:CostaLab/reg-gen@66f5fbb
hint_atac_hmm_paired,hint,footprint HMM,package
hint_atac_hmm_single,hint,footprint HMM,package


## One entry in full

`get_pretrained` returns the entry: its pinned source, files and checksums,
license, citation and how its loader is configured. Older names still
resolve as aliases (`'encode_k562'` → `'chrombpnet_encode_k562'`).

In [5]:
e = epi.models.get_pretrained('encode_k562')
print(e.name, '|', e.task, '|', e.organism, e.genome, e.biosample)
print('source :', e.source)
print('files  :', e.files)
print('url    :', e.url(next(iter(e.files))))
print('license:', e.license)
print('cite   :', e.citation)

chrombpnet_encode_k562 | sequence -> base-resolution ATAC profile + counts | human GRCh38 K562
source : {'type': 'encode', 'accession': 'ENCFF008TMU'}
files  : {'ENCFF008TMU.tar.gz': 'md5:a1b9870fcb52bba03ea5ef71c8acf897'}
url    : https://www.encodeproject.org/files/ENCFF008TMU/@@download/ENCFF008TMU.tar.gz
license: ENCODE data-use policy (free reuse with citation); chrombpnet code MIT
cite   : Pampari et al. ChromBPNet: bias factorized, base-resolution deep learning models of chromatin accessibility. bioRxiv (2024). ENCODE Project Consortium.


## Loading a model

`load_pretrained` downloads what is missing, verifies it and hands the
files to the family's loader; keyword arguments go to that loader. The
small PRINT models make the point — the Tn5 bias CNN is a TorchScript
module that maps one-hot DNA to a bias track:

In [6]:
import torch

tn5 = epi.models.load_pretrained('print_tn5_bias')
rng = np.random.default_rng(0)
seq = torch.nn.functional.one_hot(torch.from_numpy(rng.integers(0, 4, 400)), 4).T[None].float()
with torch.no_grad():
    out = tn5(seq)
print(type(tn5).__name__, tuple(seq.shape), '->', tuple(out.shape))

RecursiveScriptModule (1, 4, 400) -> (1, 300)


The same call builds non-network models too: PRINT's dispersion model
(the footprint null) and HINT-ATAC's HMM, which ships with Epione.

In [7]:
disp = epi.models.load_pretrained('print_dispersion')
hmm = epi.models.load_pretrained('hint_atac_hmm_paired')
print(type(disp).__name__, disp.version, f'{len(disp.params)} radii')
print(type(hmm).__name__, hmm.name, 'states x tracks:', np.asarray(hmm.means).shape)

DispersionModel 20250329 99 radii
HintHMM atac_paired states x tracks: (9, 8)


A seq2PRINT checkpoint trained in the [seq2PRINT tutorial](t_seq2print.ipynb)
(47 MB from Hugging Face, commit `e8a36d4`); `return_manifest=True` also
returns its training record:

In [8]:
model, manifest = epi.models.load_pretrained('seq2print_p2s1_fold0_bulk', return_manifest=True)
x = torch.nn.functional.one_hot(torch.from_numpy(rng.integers(0, 4, model.dna_len)), 4).T[None].float()
with torch.no_grad():
    footprint, coverage = model(x)
print('input', tuple(x.shape), '-> footprints', tuple(footprint.shape), '+ coverage', tuple(coverage.shape))
print('manifest records:', ', '.join(manifest))

input (1, 4, 1840) -> footprints (1, 99, 800) + coverage (1,)
manifest records: format, epione_commit, architecture, data, name, groups, training, test_metrics, history


The large models load the same way — they are not run here because each
is 0.5–0.8 GB:

```python
bias  = epi.models.load_pretrained('chrombpnet_encode_k562', kind='bias', fold=0)  # BPNet
full  = epi.models.load_pretrained('chrombpnet_encode_k562')                        # ChromBPNet
borz  = epi.models.load_pretrained('borzoi_replicate_0')        # needs borzoi-pytorch
scoob = epi.models.load_pretrained('scooby_neurips')            # needs upstream scooby
```

The scooby entries carry the constructor arguments the checkpoints need
(`cell_emb_dim`, `n_tracks`, read off the weights), so no upstream script is
needed to rebuild them:

In [9]:
for name in epi.models.list_pretrained('scooby').index:
    print(f'{name:22s}', epi.models.get_pretrained(name).loader_kwargs)

scooby_neurips         {'cell_emb_dim': 14, 'n_tracks': 3}
scooby_onek1k          {'cell_emb_dim': 10, 'n_tracks': 2}
scooby_epicardioids    {'cell_emb_dim': 50, 'n_tracks': 3}


## Cache, checksums and offline use

Files go to `EPIONE_MODELS_DIR` if set, else `EPIONE_DATA_DIR`, else the OS
cache, one directory per family — the same places the per-family loaders
always used, so earlier downloads are reused. Each verified file gets a
small `.verified` stamp (checksum, size, mtime) so a 700 MB checkpoint is
not re-hashed on every load; any change to the file invalidates it.

With `EPIONE_OFFLINE=1` (or `HF_HUB_OFFLINE=1`, or `offline=True`) nothing
is downloaded: cached files are verified and used, and a missing one raises
an error that says where it is expected. Fetch on a machine with network
(`epi.models.fetch_pretrained(name)`) and copy the cache across.

In [10]:
path = epi.models.fetch_pretrained('print_tn5_bias')
print('<cache>/' + str(path.relative_to(epi.models.pretrained_cache_dir())))
print(epi.models.registry.file_checksum(path, 'md5'), '==', epi.models.get_pretrained('print_tn5_bias').files[path.name])

epi.models.load_pretrained('print_tn5_bias', offline=True)          # cached: fine offline
try:
    epi.models.fetch_pretrained('borzoi_replicate_3', offline=True)  # not cached
except FileNotFoundError as err:
    print('offline:', str(err).split(' at ')[0], '...')

<cache>/print/Tn5_NN_model_py_v2.pt
md5:5ea6c26fcd54ad2aff3d9705ed54dd3c == md5:5ea6c26fcd54ad2aff3d9705ed54dd3c
offline: borzoi_replicate_3: config.json is not cached ...


## Registering your own model

Any trained model can be registered and then loaded by name — by you, by a
collaborator from a shared directory, or by a downstream package that
publishes weights. Here the seq2PRINT model above is saved with its
family's `save_pretrained`, which returns the checksums, and registered as
a local entry. A new model family registers its entries the same way,
passing `loader='package.module:function'`; the loader is called as
`loader(entry, **kwargs)` and fetches files with `entry.fetch(filename)`.

In [11]:
import tempfile

out = pathlib.Path(tempfile.mkdtemp()) / 'my_checkpoint'
files = epi.models.seq2print.save_pretrained(model, out, manifest={
    'architecture': manifest['architecture'], 'note': 'copy of seq2print_p2s1_fold0_bulk'})
epi.models.register({
    'name': 'my_seq2print', 'family': 'seq2print', 'task': 'sequence -> multi-scale footprints',
    'organism': 'mouse', 'genome': 'mm10', 'license': 'GPL-3.0',
    'source': {'type': 'local', 'path': str(out)}, 'files': files,
})
mine = epi.models.load_pretrained('my_seq2print')
with torch.no_grad():
    print('identical predictions:', torch.equal(mine(x)[0], footprint))
epi.models.list_pretrained(name='my_seq2print')[['family', 'source', 'files', 'cached']]

identical predictions: True


,family,source,files,cached
name,,,,
my_seq2print,seq2print,local,2,True


## Summary

| Family | Entries | Source | Loader returns |
|---|---|---|---|
| `chrombpnet` | 5 ENCODE cell lines × 5 folds × {full, nobias, bias} | ENCODE accession + md5 | `ChromBPNet` / `BPNet` (no TensorFlow) |
| `seq2print` | 2 Epione checkpoints (mouse P2 brain) | Hugging Face commit + md5 | `Seq2PRINT` (+ manifest) |
| `print` | dispersion, Tn5 bias, TF binding | Zenodo record + md5 | `DispersionModel`, TorchScript |
| `hint` | 8-mer bias tables, 2 HMMs | RGT Git commit + md5 / shipped | tables or `HintBias`, `HintHMM` |
| `borzoi` | 4 human replicates | Hugging Face commit + sha256 | `Borzoi` |
| `scooby` | NeurIPS BMMC, OneK1K, epicardioids | Hugging Face commit + sha256 | upstream `Scooby` |
| `translate`, `higashi` | your own checkpoints | local path (+ checksum) | translation model, Higashi |

Torch pickles (`translate`, `higashi`) execute code when loaded: register
only files you trust — the checksum then guarantees it is that file.